# 🧹 BƯỚC 2: Tiền xử lý dữ liệu — TOÀN BỘ DATA (8GB RAM)
**Amazon Clothing Review Analysis & Recommendation**

---
### 🎯 Mục tiêu:
Xử lý **toàn bộ 22.6M reviews** trên máy **8GB RAM** mà không bị crash.

### 💡 Chiến lược — CHUNKING:
```
Thay vì:  Load 22.6M dòng → RAM đầy → CRASH 💥

Thay bằng: Đọc 100k dòng → Xử lý → Lưu tạm
           Đọc 100k dòng → Xử lý → Lưu tạm
           ... (lặp ~226 lần)
           Ghép tất cả lại → 1 file parquet hoàn chỉnh ✅
```
Tại mỗi chunk RAM chỉ dùng ~200-300MB → an toàn với 8GB.

## Cell 2.1 — Cài thêm thư viện & Import

In [ ]:
import subprocess
subprocess.run(["pip", "install", "pyarrow", "tqdm", "psutil", "-q"])

import pandas as pd
import numpy as np
import json, gzip, gc
import matplotlib.pyplot as plt
import psutil
from pathlib import Path
from tqdm import tqdm
from collections import Counter
import pyarrow as pa
import pyarrow.parquet as pq

def ram_usage():
    vm = psutil.virtual_memory()
    return f"RAM: {vm.used/(1024**3):.1f}/{vm.total/(1024**3):.1f} GB ({vm.percent:.0f}%)"

print("✅ Libraries loaded!")
print(f"💾 {ram_usage()}")

## Cell 2.2 — Cấu hình đường dẫn & tham số

In [ ]:
ROOT_DIR      = Path().resolve().parent
REVIEW_PATH   = ROOT_DIR / "data" / "raw" / "Clothing_Shoes_and_Jewelry.jsonl.gz"
META_PATH     = ROOT_DIR / "data" / "raw" / "meta_Clothing_Shoes_and_Jewelry.jsonl.gz"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
SAMPLE_DIR    = ROOT_DIR / "data" / "sample"
FIGURES_DIR   = ROOT_DIR / "outputs" / "figures"
TEMP_DIR      = ROOT_DIR / "data" / "processed" / "_temp_chunks"

for d in [PROCESSED_DIR, SAMPLE_DIR, FIGURES_DIR, TEMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ============================================================
# CHUNKING CONFIG
# CHUNK_SIZE = 100_000 → mỗi lần đọc 100k dòng (~150MB RAM)
# Giảm xuống 50_000 nếu máy bị lag
# ============================================================
CHUNK_SIZE           = 100_000
MIN_TEXT_LENGTH      = 10
MIN_REVIEWS_PER_USER = 5
MIN_REVIEWS_PER_ITEM = 5
RANDOM_SEED          = 42
np.random.seed(RANDOM_SEED)

print("✅ Config sẵn sàng!")
print(f"   Chunk size : {CHUNK_SIZE:,} dòng/lần")
print(f"   Review file: {REVIEW_PATH.stat().st_size/(1024**3):.2f} GB")
print(f"   Meta file  : {META_PATH.stat().st_size/(1024**3):.2f} GB")
print(f"💾 {ram_usage()}")

## Cell 2.3 — Đếm tổng số dòng

In [ ]:
# Chỉ đếm dòng, KHÔNG load vào RAM
# Mất ~3-5 phút, chạy 1 lần duy nhất

print("🔢 Đang đếm số dòng (mất ~3-5 phút)...")

def count_lines(filepath):
    count = 0
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for _ in tqdm(f, desc=filepath.name[:35], unit=" lines"):
            count += 1
    return count

n_reviews = count_lines(REVIEW_PATH)
n_meta    = count_lines(META_PATH)
n_chunks  = (n_reviews // CHUNK_SIZE) + 1

print(f"\n📊 Kết quả:")
print(f"   Review: {n_reviews:,} dòng → {n_chunks} chunks")
print(f"   Meta  : {n_meta:,} dòng")
print(f"\n⏱️  Ước tính thời gian xử lý: ~{n_chunks*8//60} phút")

## Cell 2.4 — Hàm làm sạch 1 chunk review

In [ ]:
# Hàm này được gọi lặp lại ~226 lần, mỗi lần xử lý 100k dòng

def clean_review_chunk(df):
    if df.empty:
        return None

    # A. Chọn cột cần thiết
    needed = ['rating', 'text', 'title', 'user_id',
              'parent_asin', 'timestamp', 'helpful_vote', 'verified_purchase']
    df = df[[c for c in needed if c in df.columns]].copy()

    # B. Drop thiếu rating hoặc text (bắt buộc phải có)
    df = df.dropna(subset=['rating', 'text'])
    if df.empty: return None

    # C. Chuẩn hóa kiểu dữ liệu
    df['rating']       = pd.to_numeric(df['rating'], errors='coerce')
    df['helpful_vote'] = pd.to_numeric(df.get('helpful_vote', 0), errors='coerce').fillna(0).astype(int)
    df = df.dropna(subset=['rating'])
    if df.empty: return None
    df['rating'] = df['rating'].astype('float32')

    # D. Timestamp → year, month
    if 'timestamp' in df.columns:
        dt = pd.to_datetime(df['timestamp'], unit='ms', errors='coerce')
        df['year']  = dt.dt.year.astype('Int16')
        df['month'] = dt.dt.month.astype('Int8')

    # E. Tạo nhãn Sentiment
    #    4-5 sao = positive | 3 sao = neutral | 1-2 sao = negative
    df['sentiment'] = df['rating'].map(
        lambda r: 'positive' if r >= 4 else ('neutral' if r == 3 else 'negative')
    ).astype('category')

    # F. Lọc review quá ngắn (< 10 ký tự = noise)
    df['text_length'] = df['text'].astype(str).str.len().astype('int32')
    df = df[df['text_length'] >= MIN_TEXT_LENGTH]
    if df.empty: return None

    # G. Loại duplicate trong cùng chunk
    df = df.drop_duplicates(subset=['user_id', 'parent_asin'], keep='first')

    return df.reset_index(drop=True)

print("✅ Hàm clean_review_chunk sẵn sàng!")

## Cell 2.5 — Xử lý toàn bộ Review theo chunking ⚡

In [ ]:
# ============================================================
# CELL 2.5: XỬ LÝ TOÀN BỘ 22.6M REVIEWS
# ⏱️ Ước tính: 30-60 phút
#
# Tại mỗi bước:
#   RAM dùng: ~200-400MB cho chunk hiện tại
#   Các chunk trước đã được lưu xuống ổ cứng và xóa khỏi RAM
# ============================================================

# Xóa chunks cũ nếu chạy lại
for f in TEMP_DIR.glob("chunk_*.parquet"):
    f.unlink()

chunk_id      = 0
total_raw     = 0
total_clean   = 0
current_chunk = []

print(f"🚀 Bắt đầu xử lý {n_reviews:,} reviews...")
print(f"   Chunk size: {CHUNK_SIZE:,} | ~{n_chunks} chunks")
print()

with gzip.open(REVIEW_PATH, 'rt', encoding='utf-8') as f:
    pbar = tqdm(f, total=n_reviews, desc="Processing", unit=" lines")

    for line in pbar:
        total_raw += 1
        try:
            current_chunk.append(json.loads(line.strip()))
        except:
            continue

        if len(current_chunk) >= CHUNK_SIZE:
            df_chunk       = pd.DataFrame(current_chunk)
            df_chunk_clean = clean_review_chunk(df_chunk)

            if df_chunk_clean is not None:
                chunk_path = TEMP_DIR / f"chunk_{chunk_id:04d}.parquet"
                df_chunk_clean.to_parquet(chunk_path, index=False)
                total_clean += len(df_chunk_clean)

            chunk_id      += 1
            current_chunk  = []
            del df_chunk, df_chunk_clean
            gc.collect()  # Giải phóng RAM ngay lập tức

            pbar.set_postfix({
                'chunks': chunk_id,
                'clean' : f"{total_clean:,}",
                'RAM'   : f"{psutil.virtual_memory().percent:.0f}%"
            })

    # Xử lý phần dư cuối file
    if current_chunk:
        df_chunk       = pd.DataFrame(current_chunk)
        df_chunk_clean = clean_review_chunk(df_chunk)
        if df_chunk_clean is not None:
            (TEMP_DIR / f"chunk_{chunk_id:04d}.parquet").parent
            df_chunk_clean.to_parquet(TEMP_DIR / f"chunk_{chunk_id:04d}.parquet", index=False)
            total_clean += len(df_chunk_clean)
        chunk_id += 1

gc.collect()
print(f"\n✅ Hoàn tất!")
print(f"   Tổng đọc  : {total_raw:,}")
print(f"   Sau sạch  : {total_clean:,} ({total_clean/total_raw*100:.1f}% giữ lại)")
print(f"   Số chunks : {chunk_id}")
print(f"💾 {ram_usage()}")

## Cell 2.6 — Ghép tất cả chunks thành 1 file parquet

In [ ]:
# Ghép bằng pyarrow → ghi từng file một, không load hết vào RAM

chunk_files = sorted(TEMP_DIR.glob("chunk_*.parquet"))
output_path = PROCESSED_DIR / "review_clean.parquet"
print(f"📦 Ghép {len(chunk_files)} chunks...")

writer = None
for chunk_file in tqdm(chunk_files, desc="Merging chunks"):
    table = pq.read_table(chunk_file)
    if writer is None:
        writer = pq.ParquetWriter(output_path, table.schema, compression='snappy')
    writer.write_table(table)
    del table
    gc.collect()

if writer:
    writer.close()

# Xóa thư mục chunks tạm
import shutil
shutil.rmtree(TEMP_DIR)
print(f"🗑️  Đã xóa folder chunks tạm")

sz = output_path.stat().st_size / (1024**3)
print(f"\n✅ review_clean.parquet: {sz:.2f} GB")
print(f"💾 {ram_usage()}")

## Cell 2.7 — Xử lý Meta dataset

In [ ]:
# Meta nhỏ hơn nhiều so với review → load 1 lần được

def load_and_clean_meta(filepath):
    data = []
    print("📦 Loading toàn bộ meta...")
    with gzip.open(filepath, 'rt', encoding='utf-8') as f:
        for line in tqdm(f, desc="Reading", unit=" lines"):
            try:
                data.append(json.loads(line.strip()))
            except:
                continue

    df = pd.DataFrame(data)
    print(f"   Raw shape: {df.shape}")

    # Chọn cột
    needed = ['parent_asin','title','price','description',
              'categories','average_rating','rating_number','store','main_category']
    df = df[[c for c in needed if c in df.columns]].copy()

    # Drop thiếu ID, title & duplicate
    df = df.dropna(subset=['parent_asin','title'])
    df = df.drop_duplicates(subset=['parent_asin'], keep='first')

    # Price: '$29.99' → 29.99
    if 'price' in df.columns:
        df['price'] = pd.to_numeric(
            df['price'].astype(str).str.replace(r'[^\d.]','',regex=True),
            errors='coerce'
        ).astype('float32')

    # Description: list → string
    if 'description' in df.columns:
        df['description'] = df['description'].apply(
            lambda x: ' '.join(x) if isinstance(x, list) else (str(x) if pd.notna(x) else '')
        )

    # Categories → main_category (level 1)
    if 'categories' in df.columns:
        def extract_cat(cats):
            if isinstance(cats, list) and len(cats) > 0:
                inner = cats[0] if isinstance(cats[0], list) else cats
                return inner[0] if len(inner) > 0 else 'Unknown'
            return 'Unknown'
        df['main_category'] = df['categories'].apply(extract_cat)

    df = df.reset_index(drop=True)
    print(f"   Sau xử lý: {df.shape}")
    return df

df_meta_clean = load_and_clean_meta(META_PATH)

meta_path = PROCESSED_DIR / "meta_clean.parquet"
df_meta_clean.to_parquet(meta_path, index=False)
sz = meta_path.stat().st_size / (1024**2)
print(f"\n✅ meta_clean.parquet: {sz:.1f} MB")
print(f"💾 {ram_usage()}")

## Cell 2.8 — Merge Review + Meta (chunked)

In [ ]:
# Review quá lớn → merge từng chunk với meta (nhỏ, đã có trong RAM)

meta_for_merge = df_meta_clean[[
    c for c in ['parent_asin','title','price','main_category']
    if c in df_meta_clean.columns
]]

del df_meta_clean
gc.collect()

review_pf     = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
merged_output = PROCESSED_DIR / "merged_clean.parquet"
writer_m      = None
total_merged  = 0
n_rg          = review_pf.metadata.num_row_groups

print(f"🔗 Merging {n_rg} row groups với meta...")

for i in tqdm(range(n_rg), desc="Merging"):
    chunk  = review_pf.read_row_group(i).to_pandas()
    merged = chunk.merge(meta_for_merge, on='parent_asin', how='left')

    if 'title_x' in merged.columns:
        merged = merged.rename(columns={'title_x':'review_title','title_y':'product_title'})

    table = pa.Table.from_pandas(merged, preserve_index=False)
    if writer_m is None:
        writer_m = pq.ParquetWriter(merged_output, table.schema, compression='snappy')
    writer_m.write_table(table)
    total_merged += len(merged)

    del chunk, merged, table
    gc.collect()

if writer_m:
    writer_m.close()

sz = merged_output.stat().st_size / (1024**3)
print(f"\n✅ merged_clean.parquet: {total_merged:,} rows | {sz:.2f} GB")
print(f"💾 {ram_usage()}")

## Cell 2.9 — Cold-start filter cho Recommendation System

In [ ]:
# Lọc user/item có ít hơn 5 reviews (cold-start problem)
# Dùng 2 pass:
#   Pass 1: đọc hết file, đếm user/item
#   Pass 2: đọc lại, chỉ giữ user/item hợp lệ

review_pf = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
n_rg      = review_pf.metadata.num_row_groups

# === Pass 1: Đếm ===
print("Pass 1/2: Đếm user/item counts...")
user_counter = Counter()
item_counter = Counter()

for i in tqdm(range(n_rg), desc="Counting"):
    chunk = review_pf.read_row_group(i, columns=['user_id','parent_asin']).to_pandas()
    user_counter.update(chunk['user_id'].tolist())
    item_counter.update(chunk['parent_asin'].tolist())
    del chunk; gc.collect()

valid_users = {u for u,c in user_counter.items() if c >= MIN_REVIEWS_PER_USER}
valid_items = {p for p,c in item_counter.items() if c >= MIN_REVIEWS_PER_ITEM}
print(f"   Valid users: {len(valid_users):,} / {len(user_counter):,}")
print(f"   Valid items: {len(valid_items):,} / {len(item_counter):,}")

# === Pass 2: Lọc & Lưu ===
print("\nPass 2/2: Lọc và lưu...")
rec_output = PROCESSED_DIR / "review_for_rec.parquet"
writer_r   = None
total_rec  = 0

for i in tqdm(range(n_rg), desc="Filtering"):
    chunk    = review_pf.read_row_group(i).to_pandas()
    filtered = chunk[
        chunk['user_id'].isin(valid_users) &
        chunk['parent_asin'].isin(valid_items)
    ]
    if len(filtered) > 0:
        table = pa.Table.from_pandas(filtered, preserve_index=False)
        if writer_r is None:
            writer_r = pq.ParquetWriter(rec_output, table.schema, compression='snappy')
        writer_r.write_table(table)
        total_rec += len(filtered)
        del table
    del chunk, filtered; gc.collect()

if writer_r: writer_r.close()

sz = rec_output.stat().st_size / (1024**2)
print(f"\n✅ review_for_rec.parquet: {total_rec:,} rows | {sz:.1f} MB")
print(f"💾 {ram_usage()}")

## Cell 2.10 — Tạo file sample 100k để test nhanh ở bước sau

In [ ]:
# File sample nhỏ giúp test code nhanh mà không cần load hết file lớn

print("📋 Tạo file sample 100k rows...")
review_pf   = pq.ParquetFile(PROCESSED_DIR / "review_clean.parquet")
sample_rows = []
collected   = 0
target      = 100_000

for i in range(review_pf.metadata.num_row_groups):
    chunk  = review_pf.read_row_group(i).to_pandas()
    n_take = min(len(chunk), target - collected)
    sample_rows.append(chunk.sample(n=n_take, random_state=RANDOM_SEED))
    collected += n_take
    del chunk
    if collected >= target:
        break

df_sample = pd.concat(sample_rows, ignore_index=True)
df_sample.to_parquet(SAMPLE_DIR / "review_sample_100k.parquet", index=False)
print(f"✅ review_sample_100k.parquet: {len(df_sample):,} rows")

## Cell 2.11 — Báo cáo tổng kết & Visualize

In [ ]:
# Load sample 100k để vẽ biểu đồ (không cần load hết file lớn)

df_viz = pd.read_parquet(SAMPLE_DIR / "review_sample_100k.parquet")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bước 2 — Kết quả Preprocessing (toàn bộ 22.6M reviews)\nVisualize trên 100k sample',
              fontsize=13, fontweight='bold')

# 1. Sentiment distribution
sc   = df_viz['sentiment'].value_counts()
clrs = {'positive':'#2ecc71','neutral':'#f39c12','negative':'#e74c3c'}
bars = axes[0,0].bar(sc.index, sc.values, color=[clrs.get(s,'gray') for s in sc.index])
axes[0,0].set_title('Phân bố Sentiment Label')
for bar, val in zip(bars, sc.values):
    axes[0,0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                    f"{val/len(df_viz)*100:.1f}%", ha='center', fontweight='bold')

# 2. Text length
tl = df_viz['text_length'].clip(upper=1000)
axes[0,1].hist(tl, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0,1].set_title('Phân bố độ dài Review (ký tự)')
axes[0,1].axvline(tl.median(), color='red', linestyle='--',
                    label=f'Median: {tl.median():.0f}')
axes[0,1].legend()

# 3. Rating
rc = df_viz['rating'].value_counts().sort_index()
axes[1,0].bar(rc.index, rc.values,
               color=['#e74c3c','#e67e22','#f1c40f','#2ecc71','#27ae60'])
axes[1,0].set_title('Phân bố Rating (1-5 sao)')
axes[1,0].set_xlabel('Rating')

# 4. Review theo năm
if 'year' in df_viz.columns:
    yc = df_viz['year'].dropna().astype(int).value_counts().sort_index()
    yc = yc[yc.index.between(2000, 2024)]
    axes[1,1].bar(yc.index, yc.values, color='#9b59b6', alpha=0.8)
    axes[1,1].set_title('Số Review theo Năm')
    axes[1,1].tick_params(axis='x', rotation=45)

plt.tight_layout()
save_path = FIGURES_DIR / 'step2_preprocessing_result.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()

# Tổng kết
print("\n" + "="*60)
print("🎉 TỔNG KẾT BƯỚC 2")
print("="*60)
print("\n📁 FILES ĐÃ TẠO:")
for f in sorted(list(PROCESSED_DIR.glob("*.parquet")) + list(SAMPLE_DIR.glob("*.parquet"))):
    sz = f.stat().st_size
    unit = 'GB' if sz > 1024**3 else 'MB'
    val = sz/(1024**3) if sz > 1024**3 else sz/(1024**2)
    folder = 'processed' if 'processed' in str(f) else 'sample'
    print(f"  data/{folder}/{f.name:<38} {val:.1f} {unit}")

print(f"\n📊 THỐNG KÊ:")
print(f"   Tổng reviews đọc vào   : {total_raw:,}")
print(f"   Sau làm sạch            : {total_clean:,} ({total_clean/total_raw*100:.1f}% giữ lại)")
print(f"   Cho Recommendation      : {total_rec:,}")
print(f"\n🚀 Sẵn sàng BƯỚC 3: EDA chuyên sâu!")